# 03 — ML Model Evaluation
Evaluate Traffic LSTM, Energy/Water XGBoost, and Isolation Forest anomaly detector.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ml_models.model_registry import ModelRegistry
from src.ml_models.feature_engineering import (
    build_traffic_features, build_energy_features, train_test_split_temporal
)

reg = ModelRegistry('data/processed')
reg.load_all()
print(f'Loaded models: {reg.list_models()}')

In [ ]:
# Traffic LSTM evaluation
traffic_df = pd.read_parquet('data/synthetic/traffic.parquet')
seg = traffic_df[traffic_df['segment_id'] == 0].copy()
X, y = build_traffic_features(seg, window=12)
_, _, _, _, X_te, y_te = train_test_split_temporal(X, y)

if reg.is_available('traffic'):
    preds = reg.predict('traffic', X_te)
    mae  = np.mean(np.abs(preds - y_te))
    rmse = np.sqrt(np.mean((preds - y_te)**2))
    mape = np.mean(np.abs((preds - y_te) / (y_te + 1e-8))) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sample = min(500, len(preds))
    axes[0].plot(y_te[:sample], label='Actual', alpha=0.8, lw=1.2)
    axes[0].plot(preds[:sample], label='Predicted', alpha=0.8, lw=1.2, linestyle='--')
    axes[0].set_xlabel('Time Step'); axes[0].set_ylabel('Vehicle Count')
    axes[0].set_title(f'Traffic LSTM — Actual vs Predicted\nMAE={mae:.1f}  RMSE={rmse:.1f}  MAPE={mape:.1f}%')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    residuals = preds - y_te
    axes[1].hist(residuals, bins=40, color='#2196F3', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', lw=2, label='Zero error')
    axes[1].set_xlabel('Prediction Error'); axes[1].set_ylabel('Frequency')
    axes[1].set_title('Residual Distribution')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('data/processed/lstm_evaluation.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Traffic LSTM: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.1f}%')

In [ ]:
# Energy and Water XGBoost evaluation
energy_df = pd.read_parquet('data/synthetic/energy.parquet')
zone = energy_df[energy_df['zone_id'] == 0].copy()
X_e, y_e = build_energy_features(zone, window=12)
_, _, _, _, X_ete, y_ete = train_test_split_temporal(X_e, y_e)

if reg.is_available('energy'):
    e_preds = reg.predict('energy', X_ete)
    e_mae  = np.mean(np.abs(e_preds - y_ete))
    e_mape = np.mean(np.abs((e_preds - y_ete) / (y_ete + 1e-8))) * 100

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(y_ete[:300], label='Actual', lw=1.5, alpha=0.9)
    ax.plot(e_preds[:300], label='Predicted', lw=1.5, alpha=0.9, linestyle='--')
    ax.fill_between(range(300),
                     (e_preds[:300] - e_mae), (e_preds[:300] + e_mae),
                     alpha=0.2, label=f'±MAE band ({e_mae:.1f} kWh)')
    ax.set_xlabel('Time Step'); ax.set_ylabel('Energy Demand (kWh)')
    ax.set_title(f'Energy Forecaster (XGBoost) — MAE={e_mae:.1f} kWh  MAPE={e_mape:.1f}%')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('data/processed/energy_evaluation.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Anomaly detection precision / recall test
from src.anomaly_detection.detectors import TrafficAnomalyDetector
from src.ml_models.feature_engineering import build_anomaly_features
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns

det = TrafficAnomalyDetector()
det.load('data/processed/anomaly_traffic.pkl')

# Build test set with known anomalies
seg_all = traffic_df[traffic_df['segment_id'] == 0].copy()
feats = build_anomaly_features(seg_all)
labels_true = seg_all['is_anomaly'].values[:len(feats)].astype(int)

preds_raw = det.detect(feats)   # +1 = normal, -1 = anomaly
preds_bin = (preds_raw == -1).astype(int)

if labels_true.sum() > 0:
    p = precision_score(labels_true, preds_bin, zero_division=0)
    r = recall_score(labels_true, preds_bin, zero_division=0)
    f1 = f1_score(labels_true, preds_bin, zero_division=0)
    print(f'Anomaly Detector — Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}')

    cm = confusion_matrix(labels_true, preds_bin)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Anomaly'], yticklabels=['Normal','Anomaly'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix\nP={p:.2f} R={r:.2f} F1={f1:.2f}')
    plt.tight_layout()
    plt.savefig('data/processed/anomaly_evaluation.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No labelled anomalies in this segment slice.')